# Foundations: Cleaning & Exploring the Applicant Dataset

**Name:** Mian Muhammad Umair Naeem  

This notebook documents the complete cleaning pipeline for the Week 1 applicant dataset. Every major transformation is explained in Markdown before the code that implements it.

## 1. Project Objective

The goal is to transform a messy synthetic applicant dataset into a clean, trustworthy CSV using only Pandas and reproducible code. The workflow covers:

- Initial data profiling and baseline recording.
- Missing-value handling, one column at a time.
- Removal of exact and near-duplicate applicants.
- Text standardization (names, emails, universities, phones).
- Domain and status standardization.
- Date and phone type fixes.
- Final validation and export.

No manual spreadsheet edits are used anywhere.

## 2. Dataset Description

The raw file is `applicants.csv`. It has the following seven columns:

| Column | Meaning |
|--------|---------|
| `Applicant Name` | Full name of the applicant |
| `Email` | Contact email |
| `Phone` | Mobile number, kept as text to preserve leading zeros |
| `Domain Applied` | Internship track the applicant chose |
| `University` | University name |
| `Application Date` | Date the application was submitted |
| `Status` | Application outcome |

The raw data intentionally includes missing values, duplicate rows, inconsistent casing/spelling, mixed date formats, and formatting issues in phone numbers.

## 3. Import Libraries

I import **Pandas** for data manipulation and a few helper utilities.

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

RAW_CSV = Path('applicants.csv')
CLEANED_CSV = Path('applicants_cleaned.csv')
SEED = 42

## 4. Load the Raw Dataset

I read the CSV with `Phone` forced to `string`. Without this, Pandas might infer phone numbers as integers and silently drop the leading `0`.

In [2]:
df = pd.read_csv(RAW_CSV, dtype={'Phone': 'string'})

# Snapshot the raw state for the final before/after comparison
raw_shape = df.shape
raw_missing = df.isnull().sum().to_dict()
raw_row_count = raw_shape[0]
raw_col_count = raw_shape[1]
raw_duplicate_rows = df.duplicated().sum()
raw_domain_variations = df['Domain Applied'].nunique()

print(f"Raw shape: {raw_shape}")

Raw shape: (87, 7)


## 5. Initial Dataset Inspection

Before touching the data, I look at its shape, content, types, missing values, and duplicates.

In [3]:
print("Shape:")
print(df.shape)

print(" Head:")
display(df.head())

print("Random Sample:")
display(df.sample(5, random_state=SEED))

print("Info")
df.info()

print("Describe (include='all')")
display(df.describe(include='all'))

print("Missing Values")
display(df.isnull().sum())

print("Exact Duplicate Rows")
print(df.duplicated().sum())

print("Unique Values per Column")
print(df.nunique())

Shape:
(87, 7)
 Head:


,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Khadija Khan,khadija.khan1@example.com,0325108603,Data Science,Lahore University of Management Sciences,2026-05-12,Rejected
1,Naveed Ali,naveed.ali2@example.com,0348078673,Web Development,FAST NUCES,2025-03-28,Under Review
2,Ali Qureshi,ali.qureshi3@example.com,0341445199,Graphic Design,COMSATS University Islamabad,"Jan 20, 2025",Selected
3,Ali Farooqi,NaN,0345667265,digital-marketing,FAST NUCES,02/08/2025,Rejected
4,Mehwish Javed,mehwish.javed5@example.com,0323608513,data science,NED,2025-07-10,Under Review


Random Sample:


,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
76,Faisal Ali,FAISAL.ALI77@EXAMPLE.COM,0324934149,DATA SCIENCE,NED,19-May-2026,Rejected
0,Khadija Khan,khadija.khan1@example.com,0325108603,Data Science,Lahore University of Management Sciences,2026-05-12,Rejected
26,Shabina Farooqi,shabina.farooqi27@example.com,0305159166,Data Science,NUCES,15-Dec-2025,Under Review
22,Rabia Malik,rabia.malik23@example.com,0305041154,GRAPHIC DESIGN,NUCES,2026-05-20,Rejected
12,Bilal Khalid,bilal.khalid13@example.com,0327730428,DATA SCIENCE,NUCES,25/01/2025,Rejected


Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87 entries, 0 to 86
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Applicant Name    86 non-null     object
 1   Email             85 non-null     object
 2   Phone             87 non-null     string
 3   Domain Applied    87 non-null     object
 4   University        85 non-null     object
 5   Application Date  87 non-null     object
 6   Status            85 non-null     object
dtypes: object(6), string(1)
memory usage: 4.9+ KB
Describe (include='all')


,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
count,86,85,87,87,85,87,85
unique,81,83,81,24,23,81,3
top,Khadija Khan,khadija.khan1@example.com,0325108603,Data Science,NUCES,2026-05-12,Under Review
freq,2,2,2,8,9,2,29


Missing Values


Applicant Name      1
Email               2
Phone               0
Domain Applied      0
University          2
Application Date    0
Status              2
dtype: int64

Exact Duplicate Rows
2
Unique Values per Column
Applicant Name      81
Email               83
Phone               81
Domain Applied      24
University          23
Application Date    81
Status               3
dtype: int64


## 6. Data-Quality Baseline

I capture the initial state in variables so the final summary can compare raw and cleaned states.

In [4]:
baseline = {
    'rows': df.shape[0],
    'columns': df.shape[1],
    'missing_total': df.isnull().sum().sum(),
    'missing_by_column': df.isnull().sum().to_dict(),
    'exact_duplicates': int(df.duplicated().sum()),
    'domain_variations': int(df['Domain Applied'].nunique()),
    'status_variations': sorted(df['Status'].dropna().astype(str).str.strip().str.lower().unique().tolist()),
    'date_dtype': str(df['Application Date'].dtype),
    'phone_dtype': str(df['Phone'].dtype),
}

print(baseline)

{'rows': 87, 'columns': 7, 'missing_total': np.int64(7), 'missing_by_column': {'Applicant Name': 1, 'Email': 2, 'Phone': 0, 'Domain Applied': 0, 'University': 2, 'Application Date': 0, 'Status': 2}, 'exact_duplicates': 2, 'domain_variations': 24, 'status_variations': ['rejected', 'selected', 'under review'], 'date_dtype': 'object', 'phone_dtype': 'string'}


## 7. Missing-Value Handling (Column by Column)

Missing values are counted per column. I handle each column separately because the right action depends on what the column represents.

### 7.1 Applicant Name
A row without an applicant name is not usable for outreach or duplicate matching, so I drop it.

### 7.2 Email
Email is critical for contact and duplicate detection. Using a single placeholder for many rows would create false duplicate matches, so dropping is safer.

### 7.3 University
University is useful context but not essential. Missing values are filled with `Not Specified`.

### 7.4 Status
An unrecorded status most likely means the application has not been processed yet. I fill with `Under Review`.

### 7.5 Phone
Phone numbers are retained when other critical information is present. They are left as missing strings.

### 7.6 Application Date
Invalid dates are handled later by `pd.to_datetime(..., errors='coerce')`.

In [5]:
# Drop rows with missing/blank Applicant Name
name_missing_count = df['Applicant Name'].isnull().sum()
df = df.dropna(subset=['Applicant Name']).copy()
blank_name_mask = df['Applicant Name'].astype(str).str.strip() == ''
name_blank_count = blank_name_mask.sum()
df = df[~blank_name_mask].copy()
name_dropped = int(name_missing_count) + int(name_blank_count)

# Drop rows with missing/blank Email
email_missing_count = df['Email'].isnull().sum()
df = df.dropna(subset=['Email']).copy()
blank_email_mask = df['Email'].astype(str).str.strip() == ''
email_blank_count = blank_email_mask.sum()
df = df[~blank_email_mask].copy()
email_dropped = int(email_missing_count) + int(email_blank_count)

# Fill missing/blank University
university_filled = df['University'].isnull().sum() + (df['University'].astype(str).str.strip() == '').sum()
df['University'] = df['University'].fillna('Not Specified')
df.loc[df['University'].astype(str).str.strip() == '', 'University'] = 'Not Specified'

# Fill missing/blank Status
status_filled = df['Status'].isnull().sum() + (df['Status'].astype(str).str.strip() == '').sum()
df['Status'] = df['Status'].fillna('Under Review')
df.loc[df['Status'].astype(str).str.strip() == '', 'Status'] = 'Under Review'

# Keep Phone as string
df['Phone'] = df['Phone'].astype('string')

print(f"Rows remaining after missing-value handling: {df.shape[0]}")

Rows remaining after missing-value handling: 84


## 8. Duplicate Handling

### 8.1 Exact Duplicates
Two rows that are identical across every column are considered exact duplicates. `df.duplicated()` counts them, and `drop_duplicates()` removes them.

### 8.2 Near-Duplicates
The same applicant may apply twice with slightly different formatting. Email is a more reliable unique identifier than name because two people can share the same name but not the same email. I create a normalized email key (`strip` + `lower`) and keep the first occurrence.

In [6]:
# Exact duplicates
exact_duplicate_count = df.duplicated().sum()
df = df.drop_duplicates().copy()

# Near-duplicates by normalized email
df['email_norm'] = df['Email'].astype(str).str.strip().str.lower()
near_duplicate_count = df['email_norm'].duplicated().sum()
df = df.drop_duplicates(subset=['email_norm'], keep='first').copy()
df = df.drop(columns=['email_norm'])

print(f"Exact duplicates removed: {exact_duplicate_count}")
print(f"Near-duplicates removed: {near_duplicate_count}")
print(f"Rows after duplicate removal: {df.shape[0]}")

Exact duplicates removed: 2
Near-duplicates removed: 3
Rows after duplicate removal: 79


## 9. Text-Field Standardization

I clean leading/trailing whitespace, collapse internal whitespace, and apply consistent casing.

In [7]:
# Applicant Name: strip, collapse spaces, title case
df['Applicant Name'] = (df['Applicant Name']
                        .astype(str)
                        .str.strip()
                        .str.replace(r'\s+', ' ', regex=True)
                        .str.title())

# Email: strip and lowercase
df['Email'] = df['Email'].astype(str).str.strip().str.lower()

print("Sample names after cleaning:")
print(df['Applicant Name'].head())

Sample names after cleaning:
0      Khadija Khan
1        Naveed Ali
2       Ali Qureshi
4     Mehwish Javed
5    Kamran Hussain
Name: Applicant Name, dtype: object


## 10. Domain and Status Standardization

The raw `Domain Applied` column contains multiple spellings and casings for the same five internship tracks. I map every variant to a canonical label.

All status values are normalized to one of three canonical outcomes.

In [8]:
DOMAIN_MAP = {
    'web dev': 'Web Development',
    'webdev': 'Web Development',
    'web development': 'Web Development',
    'data science': 'Data Science',
    'data-science': 'Data Science',
    'datasci': 'Data Science',
    'machine learning': 'Machine Learning',
    'machine-learning': 'Machine Learning',
    'ml': 'Machine Learning',
    'graphic design': 'Graphic Design',
    'graphics design': 'Graphic Design',
    'graphicdesign': 'Graphic Design',
    'digital marketing': 'Digital Marketing',
    'digital-marketing': 'Digital Marketing',
    'digitalmark': 'Digital Marketing',
}

df['Domain Applied'] = (df['Domain Applied']
                        .astype(str)
                        .str.strip()
                        .str.lower()
                        .map(DOMAIN_MAP))

STATUS_MAP = {
    'under review': 'Under Review',
    'selected': 'Selected',
    'rejected': 'Rejected',
}

df['Status'] = (df['Status']
                .astype(str)
                .str.strip()
                .str.lower()
                .map(STATUS_MAP))

if df['Status'].isnull().any():
    df['Status'] = df['Status'].fillna('Under Review')

print("Domain value counts:")
print(df['Domain Applied'].value_counts())
print("Status value counts:")
print(df['Status'].value_counts())

Domain value counts:
Domain Applied
Digital Marketing    18
Data Science         17
Machine Learning     16
Graphic Design       14
Web Development      14
Name: count, dtype: int64
Status value counts:
Status
Under Review    29
Selected        27
Rejected        23
Name: count, dtype: int64


## 11. Date and Phone Normalization

`Application Date` is stored as text with mixed formats. I use `pd.to_datetime` with `errors='coerce'` so unreadable dates become `NaT`. Because slash-separated dates are `dd/mm/yyyy`, we pass `dayfirst=True`. The `format='mixed'` option (pandas >= 2.0) lets pandas parse each date string independently.

Phone numbers are kept as text. I remove spaces, dashes, and parentheses while preserving leading zeros.

In [9]:
# format='mixed' is available in pandas >= 2.0 and parses mixed date formats per row.
df['Application Date'] = pd.to_datetime(df['Application Date'], errors='coerce', dayfirst=True, format='mixed')
invalid_dates = df['Application Date'].isnull().sum()
print(f"Application Date dtype: {df['Application Date'].dtype}")
print(f"Invalid dates (NaT): {invalid_dates}")

df['Phone'] = df['Phone'].astype('string').str.replace(r'[^0-9]', '', regex=True)
print(f"Phone dtype after cleaning: {df['Phone'].dtype}")

Application Date dtype: datetime64[ns]
Invalid dates (NaT): 1
Phone dtype after cleaning: string


## 12. University Normalization

Common abbreviations are mapped to their full names through an explicit dictionary. Unknown names are left as-is.

In [10]:
UNIVERSITY_MAP = {
    'FAST': 'FAST NUCES',
    'NUCES': 'National University of Computer and Emerging Sciences',
    'LUMS': 'Lahore University of Management Sciences',
    'COMSATS': 'COMSATS University Islamabad',
    'GIKI': 'Ghulam Ishaq Khan Institute',
    'NED': 'NED University',
    'UET': 'University of Engineering and Technology Lahore',
    'UET Lahore': 'University of Engineering and Technology Lahore',
    'IST': 'Institute of Space Technology',
    'NUST': 'National University of Sciences & Technology',
    'PU': 'University of the Punjab',
}

df['University'] = (df['University']
                    .astype(str)
                    .str.strip()
                    .map(lambda x: UNIVERSITY_MAP.get(x, x)))
df['University'] = (df['University']
                    .astype(str)
                    .str.replace(r'\s+', ' ', regex=True)
                    .str.strip())
df.loc[df['University'] == '', 'University'] = 'Not Specified'

print("Unique universities after cleaning:")
print(df['University'].unique())

Unique universities after cleaning:
['Lahore University of Management Sciences' 'FAST NUCES'
 'COMSATS University Islamabad' 'NED University'
 'Ghulam Ishaq Khan Institute'
 'National University of Sciences & Technology'
 'National University of Computer and Emerging Sciences'
 'University of Engineering and Technology Lahore' 'University of Karachi'
 'Institute of Space Technology' 'University of the Punjab'
 'Not Specified']


## 13. Export Cleaned Dataset

I export the final DataFrame with `index=False` to avoid an extra unnamed index column.

In [11]:
df.to_csv(CLEANED_CSV, index=False)

# Verify by reading back
df_check = pd.read_csv(CLEANED_CSV, dtype={'Phone': 'string'})
print(f"Re-read CSV shape: {df_check.shape}")
assert df_check.shape == df.shape, "Exported CSV shape mismatch!"
assert 'Unnamed: 0' not in df_check.columns, "Unwanted index column found!"
print("Export verification: OK")

Re-read CSV shape: (79, 7)
Export verification: OK


## 14. Before-and-After Comparison

In [13]:
comparison = {
    'Metric': ['Rows', 'Columns', 'Exact Duplicate Rows', 'Domain Variations', 'Status Variations', 'Date dtype'],
    'Before': [raw_row_count, raw_col_count, raw_duplicate_rows, raw_domain_variations,
               len(baseline['status_variations']), baseline['date_dtype']],
    'After': [df.shape[0], df.shape[1], df.duplicated().sum(), df['Domain Applied'].nunique(),
              len(df['Status'].unique()), str(df['Application Date'].dtype)]
}

comparison_df = pd.DataFrame(comparison)
display(comparison_df)


,Metric,Before,After
0,Rows,87,79
1,Columns,7,7
2,Exact Duplicate Rows,2,0
3,Domain Variations,24,5
4,Status Variations,3,3
5,Date dtype,object,datetime64[ns]


| Metric | Before | After |
|--------|--------|--------|
| Rows | 87 | 79 |
| Columns | 7 | 7 |
| Exact Duplicate Rows | 2 | 0 |
| Domain Variations | 24 | 5 |
| Status Variations | 3 | 3 |
| Date dtype | object | datetime64[ns] |

## 15. Data Quality Summary

### Starting Dataset
- **Rows:** 87
- **Columns:** 7

### Issues Found
- Missing values by column (raw): `{'Applicant Name': 1, 'Email': 2, 'Phone': 0, 'Domain Applied': 0, 'University': 2, 'Application Date': 0, 'Status': 2}`
- Exact duplicate rows: 2
- Near-duplicate applicant records by normalized email: 3
- Raw domain variations: 24
- Invalid/unreadable dates (converted to NaT): 1
- Inconsistent phone formats and leading-zero risk: present in raw data
- Inconsistent status values (casing/typos): present in raw data

### Actions Taken
- Dropped 1 row(s) with missing/blank Applicant Name.
- Dropped 2 row(s) with missing/blank Email.
- Filled 2 missing/blank University value(s) with "Not Specified".
- Filled 2 missing/blank Status value(s) with "Under Review".
- Removed 2 exact duplicate row(s).
- Removed 3 near-duplicate row(s) by normalized email.
- Standardized Applicant Name, Email, University, and Phone text fields.
- Mapped raw Domain Applied values to exactly 5 canonical categories.
- Normalized Status to `Under Review` / `Selected` / `Rejected`.
- Converted Application Date to datetime using `pd.to_datetime(..., errors='coerce', dayfirst=True, format='mixed')`.
- Cleaned Phone numbers by removing non-digit characters and keeping them as strings.

### Final Dataset
- **Rows:** 79
- **Columns:** 7
- **Remaining missing values:** 1
- **Remaining exact duplicates:** 0
- **Canonical domains:** ['Data Science', 'Digital Marketing', 'Graphic Design', 'Machine Learning', 'Web Development']
- **Final statuses:** ['Rejected', 'Selected', 'Under Review']
- **Final date dtype:** datetime64[ns]
- **Final phone dtype:** string
- **Validation status:** ALL PASS

## 16. Conclusion

The raw applicant dataset has been cleaned entirely through code. Key outcomes:

- Missing values were handled on a per-column basis, with critical fields (name, email) causing row removal and non-critical fields filled with documented defaults.
- Exact and near-duplicates were removed using full-row comparison and normalized email.
- Text fields were standardized, domains were mapped to five canonical categories, and statuses were normalized.
- Application Date became a true `datetime` and phones remained strings with leading zeros.
- A final validation table confirmed all cleaning requirements were met.

The cleaned data is saved as `applicants_cleaned.csv`.